# Visualizing loss functions trends

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = False

## Risk definitions, as function of v

In [ ]:
def ln_clamped(x, A=-16):
    eps = np.exp(A)
    return np.log(np.clip(x, a_min=eps, a_max=None))


In [ ]:
def r_CE(v, eta, K, ln_fn = np.log) -> float or np.ndarray:
    return - (1 - eta) * ln_fn(v) - eta* ln_fn((1 - v) / (K - 1))

def r_MAE(v, eta, K) -> float or np.ndarray:
    return 2 * (K / (K - 1) * eta - 1) * v - 2 / (K - 1) * eta + 2

def r_GCE(v, eta, K, q) -> float or np.ndarray:
    return (1 - eta) / q * (1 - v**q) + eta / q * (1 - (1 - v)**q / (K - 1)**q)


## Risk plots

In [ ]:
K = [2, 3, 5, 10]
etas = [0, 0.2, 0.3, 0.4, 0.6]

risk_fns = [
    lambda v, eta, K: r_CE(v, eta, K, ln_fn=ln_clamped), 
    lambda v, eta, K: r_GCE(v, eta, K, q=0.4), 
    lambda v, eta, K: r_GCE(v, eta, K, q=0.7), 
    r_MAE]
risk_fn_names = ["Cross-Entropy", "GCE q=0.4", "GCE q=0.7", "MAE"]
n_risk = len(risk_fns)

# Plot parameters
K_colors = plt.cm.gray_r(np.linspace(0.3, 1, len(K)))
annotation_margin = 0.02

fig, axs = plt.subplots(n_risk, len(etas), figsize=(5*len(etas), 3*n_risk))
for i, risk_fn in enumerate(risk_fns):
    for j, e in enumerate(etas):
        v_mins = []
        for k in K:
            v = np.linspace(1 / k, 1, 200)
            r = risk_fn(v, e, k)
            # Get minimal point
            r_min_index = np.argmin(r)
            v_min = v[r_min_index]
            # Plot the risk function and the minimum point
            axs[i, j].plot(v, r, label=f"K={k}", color=K_colors[K.index(k)])
            vmin_text = r"$v_{min}$"
            axs[i, j].scatter(v_min, r[r_min_index], label=rf"{vmin_text}={v_min:.2f}", color='red',zorder=5)

            v_mins.append(v_min)
        # Annotate the optimal v on the x axis, angle the annotation to avoid overlap
        # if len(set([round(vi, 2) for vi in v_mins])) == 1:
        #     print(i, j)
        #     # axs[i, j].annotate(text=f"{v_mins[0]:.2f}", xy=(v_mins[0], 0), color="red", xycoords="data", textcoords="offset points", ha='center', va='bottom', xytext=(0, -10))
        # else:
        #     for k, v_min in zip(K, v_mins):
        #         # axs[i, j].annotate(text=f"{v_min:.2f}", xy=(v_min, risk(v_min, e, k)), color="red", xytext=(v_min+annotation_margin, risk(v_min, e, k)+annotation_margin))
        #         axs[i, j].set_xticks(list(axs[i, j].get_xticks()) + [v_min], labels=[f"{tick:.2f}" for tick in list(axs[i, j].get_xticks()) + [v_min]], rotation=45)
        axs[i, j].set_title(rf"Risk function {risk_fn_names[i]} with $\eta={e}$")
        axs[i, j].set_xlabel("v")
        axs[i, j].set_ylabel("Risk")
        axs[i, j].legend(ncol=len(K))
        
plt.tight_layout()
plt.show()

In [ ]:
# Same but on the same graph for a given K
K_solo = 5

for j, e in enumerate(etas):
    fig, axs = plt.subplots(1, 1, figsize=(6, 4))
    for i, risk_fn in enumerate(risk_fns):
        v = np.linspace(1 / K_solo, 1, 200)
        r = risk_fn(v, e, K_solo)
        # Get minimal point
        r_min_index = np.argmin(r)
        v_min = v[r_min_index]
        # Plot the risk function and the minimum point
        axs.plot(v, r, label=f"{risk_fn_names[i]}")
        vmin_text = r"$v_{min}$"
        axs.set_title(rf"Risk functions with $\eta={e}$ and $K={K_solo}$")
        axs.set_xlabel("v")
        axs.set_ylabel("Risk")
        
        axs.scatter(v_min, r[r_min_index], color='red',zorder=5)
        axs.legend()
        
        # axs.scatter(v_min, r[r_min_index], label=rf"{vmin_text}={v_min:.2f}", color='red',zorder=5)
        # axs.legend(ncol=len(risk_fns))
    plt.show()

## Minima expressions

In [ ]:
def v_min_CE(eta, K) -> float or np.ndarray:
    return (1 - eta) * (eta < (K-1)/K) + 1/K * (eta >= (K-1)/K)

def v_min_MAE(eta, K) -> float or np.ndarray:
    return 1 *(eta <= (K - 1)/K) # if eta <= (K - 1)/K, v_min = 0, else v_min = 1 (boolean to keep vectorization)

def v_min_GCE(eta, K, q) -> float or np.ndarray:
    if q == 1:
        return 1 *(eta <= (K - 1)/K) + 1/K * (eta > (K - 1)/K) # if eta <= (K - 1)/K, v_min = 0, else v_min = 1 (boolean to keep vectorization)
    return 1 / (1 + (K - 1) ** (-q / (1 - q)) * (eta / (1 - eta)) ** (1 / (1 - q))) * (eta <= (K - 1)/K) + 1/K * (eta > (K - 1)/K)


## Plot minima, function of noise rate

In [ ]:
print("Tested K:", K)
# vmin_fns = [v_min_CE, lambda eta, K: v_min_GCE(eta, K, q=0.4), lambda eta, K: v_min_GCE(eta, K, q=0.8), lambda eta, K: v_min_MAE(eta, K)]
# vmin_names = ["CE", r"GCE $q=0.4$", r"GCE $q=0.8$", "MAE"]
vmin_fns = [v_min_CE, lambda eta, K: v_min_GCE(eta, K, q=0.4), lambda eta, K: v_min_GCE(eta, K, q=0.8), lambda eta, K: v_min_GCE(eta, K, q=1.0)]
vmin_names = ["CE", r"GCE $q=0.4$", r"GCE $q=0.8$", r"GCE $q=1.0$ (MAE)"]
# vmin_fns = [v_min_CE, lambda eta, K: v_min_GCE(eta, K, q=0.1), lambda eta, K: v_min_GCE(eta, K, q=0.4), lambda eta, K: v_min_GCE(eta, K, q=0.8), lambda eta, K: v_min_GCE(eta, K, q=1.0)]
# vmin_names = ["CE", r"GCE $q=0.1$", r"GCE $q=0.4$", r"GCE $q=0.8$", r"GCE $q=1.0$ (MAE)"]
n_vmin = len(vmin_fns)

# Plot parameters
n_samples_eta = 1000

fig, axs = plt.subplots(1, n_vmin, figsize=(3*n_vmin, 3))
for k in K:
    etas_low = np.array(list(np.linspace(0, (k - 1) / k, int(n_samples_eta * (k - 1) / k))) + [(k - 1) / k])
    etas_high = np.linspace((k - 1) / k, 1, int(n_samples_eta * (1 - (k - 1) / k)))[1:]
    etas_list = np.concatenate([etas_low, etas_high])
    
    for i, vmin_fn in enumerate(vmin_fns):
        vmins_low = vmin_fn(etas_low, k)
        vmins_high = vmin_fn(etas_high, k)
        # axs[i].plot(etas_low, vmins_low, label=f"K={k}", color=K_colors[K.index(k)])
        # axs[i].plot(etas_high, vmins_high, color=K_colors[K.index(k)])
        axs[i].plot(etas_list, np.concatenate([vmins_low, vmins_high]), label=r"$K=" + str(k) + r"$", color=K_colors[K.index(k)])
        # axs[i].scatter([(k - 1) / k], [1 / k], color=K_colors[K.index(k)], marker="x")
        
        if k == K[0]:
            # 1/K line
            # axs[i].axhline(1 / k, color=K_colors[K.index(k)], alpha=0.3, linestyle='--')
            axs[i].annotate(text=r"$\frac{1}{K}$", 
                            xy=(0.7, 1/k), 
                            xytext=(0.9, 1/k + 0.1), 
                            arrowprops=dict(arrowstyle='->', color=K_colors[K.index(k)]),
                            fontsize=14
                            )
            # K-1/K line
            # axs[i].axvline((k - 1)/k, color=K_colors[K.index(k)], alpha=0.3, linestyle='--')
            axs[i].annotate(text=r"$\frac{K-1}{K}$", 
                            xytext=((k-1)/k - 0.4, 0.05), 
                            xy=((k-1)/k, 0.1), 
                            arrowprops=dict(arrowstyle='->', color=K_colors[K.index(k)]), 
                            fontsize=14
                            )
        
        # axs[i].set_title(rf"Optimal $v$ for {vmin_names[i]}")
        axs[i].set_title(rf"{vmin_names[i]}")
        axs[i].set_xlabel(r"$\eta$")
        axs[i].set_ylabel(r"$v^*$")
        axs[i].fill_between([(k - 1) / k, 1], [1 / k], color=K_colors[K.index(k)], alpha=0.3)
        
axs[i].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Save svg
plt.savefig("loss_fn_minimization.svg")

plt.tight_layout()
plt.show()

In [ ]:
print("Tested K:", K)
# vmin_fns = [v_min_CE, lambda eta, K: v_min_GCE(eta, K, q=0.4), lambda eta, K: v_min_GCE(eta, K, q=0.8), lambda eta, K: v_min_MAE(eta, K)]
# vmin_names = ["CE", r"GCE $q=0.4$", r"GCE $q=0.8$", "MAE"]
vmin_fns = [v_min_CE, lambda eta, K: v_min_GCE(eta, K, q=0.4), lambda eta, K: v_min_GCE(eta, K, q=0.8), lambda eta, K: v_min_GCE(eta, K, q=1.0)]
vmin_names = ["CE", r"GCE $q=0.4$", r"GCE $q=0.8$", r"GCE $q=1.0$ (MAE)"]
# vmin_fns = [v_min_CE, lambda eta, K: v_min_GCE(eta, K, q=0.1), lambda eta, K: v_min_GCE(eta, K, q=0.4), lambda eta, K: v_min_GCE(eta, K, q=0.8), lambda eta, K: v_min_GCE(eta, K, q=1.0)]
# vmin_names = ["CE", r"GCE $q=0.1$", r"GCE $q=0.4$", r"GCE $q=0.8$", r"GCE $q=1.0$ (MAE)"]
n_vmin = len(vmin_fns)

# Plot parameters
n_samples_eta = 1000

fig, axs = plt.subplots(1, n_vmin, figsize=(3*n_vmin, 3))
for k in K:
    etas_low = np.array(list(np.linspace(0, (k - 1) / k, int(n_samples_eta * (k - 1) / k))) + [(k - 1) / k])
    etas_high = np.linspace((k - 1) / k, 1, int(n_samples_eta * (1 - (k - 1) / k)))[1:]
    etas_list = np.concatenate([etas_low, etas_high])
    
    for i, vmin_fn in enumerate(vmin_fns):
        vmins_low = vmin_fn(etas_low, k)
        vmins_high = vmin_fn(etas_high, k)
        axs[i].plot(etas_low, vmins_low, label=f"K={k}", color=K_colors[K.index(k)])
        axs[i].plot(etas_high, vmins_high, "--", color=K_colors[K.index(k)])
        axs[i].scatter([(k - 1) / k], [vmin_fn((k - 1) / k, k)], color=K_colors[K.index(k)], marker="x")
        axs[i].fill_between([(k - 1) / k, 1], [1 / k], color=K_colors[K.index(k)], alpha=0.3)
        
        # axs[i].set_title(rf"Optimal $v$ for {vmin_names[i]}")
        axs[i].set_title(rf"{vmin_names[i]}")
        axs[i].set_xlabel(r"$\eta$")
        axs[i].set_ylabel(r"$v^*$")
        
axs[i].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
plt.tight_layout()
plt.show()

## Minimum simulation to check v_min formulas

In [ ]:
# Plot the optimal v as a function of eta for each risk function and K
n_samples_eta = 101 # Horizontal resolution
n_samples_v = 1000    # Vertical resolution


fig, axs = plt.subplots(1, n_risk, figsize=(4*n_risk, 3))
for k in K:
    etas_list = np.array(list(np.linspace(0, 1, n_samples_eta)))

    for i, risk_fn in enumerate(risk_fns):
        vmins_list = []
        for eta in etas_list:
            # compute min of risk function for each eta and k
            v = np.linspace(0, 1, n_samples_v)
            r = risk_fn(v, eta, k)
            # Get minimal point
            r_min_index = np.argmin(r)
            vmins_list.append(v[r_min_index])

        print(risk_fn_names[i], k)
        print(etas_list, vmins_list)
        axs[i].plot(etas_list, vmins_list, label=f"K={k}", color=K_colors[K.index(k)])
        axs[i].set_title(f"Optimal v for {risk_fn_names[i]}") # as a function of eta")
        axs[i].set_xlabel(r"$\eta$")
        axs[i].set_ylabel(r"$v^*$")
        axs[i].legend()
        
        # Special ranges
        axs[i].fill_between([(k - 1) / k, 1], [1 / k], color=K_colors[K.index(k)], alpha=0.3)
        if k == K[0]:
            # 1/K line
            axs[i].axhline(1 / k, color=K_colors[K.index(k)], alpha=0.3, linestyle='--')
            axs[i].annotate(text=f"1/K", xy=(0.5, 1/k), xytext=(0.3, 1/k - 0.1), arrowprops=dict(arrowstyle='->', color=K_colors[K.index(k)]))
            # K-1/K line
            axs[i].axvline((k - 1)/k, color=K_colors[K.index(k)], alpha=0.3, linestyle='--')
            axs[i].annotate(text=f"(K-1)/K", xy=((k-1)/k, 0.1), xytext=((k-1)/k - 0.2, 0.05), arrowprops=dict(arrowstyle='->', color=K_colors[K.index(k)]))

plt.tight_layout()
plt.show()